## Prime95B Post-Analysis — Filter, Undrift, Link

Mono equivalent of `Ximea_PostAnalysis.ipynb`, adapted for the NOCOLOUR schema
produced by `Raw_Analysis.ipynb` (`<FOV>_nocolour_fit.h5`: `xc, yc, s_x, s_y,
bg, A, chi_sqr, frame` + `_err` columns + `photons`/`background_photons`).

- **Filter**: `SM_extractionfunctions.filter_quality_localisations` (used by
  `extract_single_molecules_HDBSCAN`) unconditionally requires
  `A_R_err/A_G_err/A_B_err/bg_R_err/bg_G_err/bg_B_err`
  (`src/SM_extractionfunctions.py:130-131`), which don't exist for mono data —
  so filtering is a bespoke pandas threshold chain here, using only the
  columns that actually exist.
- **AIM undrift**: identical call to the Ximea notebook (AIM only touches
  `xc`/`yc`/`frame`, never colour columns) — just Prime95B's own pixel size.
- **Link with HDBSCAN**: a small local helper reusing the same
  `fast_hdbscan`/`sklearn` clustering call and `cluster_selection_epsilon`
  convention as `src/clustering/hdbscan_clusterer.py`, then reusing the
  *existing* `SM_extractionfunctions.extract_SMs().average_parameters()` for
  the per-cluster averaging step — that method is colour-agnostic (only
  special-cases `A_B/A_G/A_R` if present; everything else gets a plain mean),
  so no duplication needed there.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path

import sys
sys.path.append('../..')

from src import IOFunctions
import DriftCorrectionFunctions as DCF
from src import SM_extractionfunctions

# fast_hdbscan preferred (matches src/clustering/hdbscan_clusterer.py); falls back to sklearn.
try:
    from fast_hdbscan import HDBSCAN
except ImportError:
    from sklearn.cluster import HDBSCAN

IO = IOFunctions.IO_Functions()
# camera='ximea' here is only a placeholder to satisfy the constructor's camera
# registry validation (Prime95B isn't registered — see Raw_Analysis.ipynb) —
# only the colour-agnostic .average_parameters() method is used from this
# instance, which doesn't depend on self.pixel_size/camera at all.
SM_E = SM_extractionfunctions.extract_SMs(camera='ximea')


In [ ]:
# ── Paths and acquisition parameters ────────────────────────────────────────────
PRIME95B_FOLDER = Path('/scratch/sycamore_asap_server/ASAP_Members_Other_Imaging_Data/Brendan/20260624_Ximea_vs_Prime95_beads/100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Prime95B')

PIXEL_SIZE_NM = 110.0
WIDTH, HEIGHT = 1200, 1200   # Prime95B chip shape (confirmed via Camera_Calibrations/Prime95B_Camera/gain.tif)

# AIM drift correction — same convention as Ximea, scaled by this camera's pixel size
AIM_SEGMENTATION = 20
AIM_INTERSECT_D  = 20 / PIXEL_SIZE_NM
AIM_ROI_R        = 60 / PIXEL_SIZE_NM

# HDBSCAN linking
MIN_CLUSTER_SIZE = 10

prime95b_files = sorted(PRIME95B_FOLDER.glob('*_nocolour_fit.h5'))
print(f'[Prime95B] {len(prime95b_files)} FOV .h5 files found')


### Preview — tune filter/undrift/link parameters on one FOV

In [ ]:
# ── Filtering thresholds — mono subset of Constants.FilteringConstants ─────────
# Same starting values as the Ximea notebook, minus every colour-fraction/colour-error
# term (A_B/A_G/A_R and bg_B/bg_G/bg_R don't exist for NOCOLOUR data).
from src.Constants import FilteringConstants

MAX_LOCALISATION_ERROR_PX = FilteringConstants.MAX_LOCALISATION_ERROR_PX   # 1.0 px
MIN_SIGMA_PX               = FilteringConstants.MIN_SIGMA_NM / PIXEL_SIZE_NM
MAX_SIGMA_PX               = FilteringConstants.MAX_SIGMA_NM / PIXEL_SIZE_NM
MAX_SIGMA_ERROR_PX         = FilteringConstants.MAX_SIGMA_ERROR_NM / PIXEL_SIZE_NM
MIN_PHOTONS                = FilteringConstants.MIN_PHOTONS


def filter_prime95b_locs(df):
    """Mono equivalent of filter_quality_localisations — same structure, only
    the columns that actually exist for the NOCOLOUR schema."""
    chi_val = np.median(df['chi_sqr'])
    df = df[df['chi_sqr'] < chi_val]
    df = df[(df['xc_err'] > 0) & (df['xc_err'] < MAX_LOCALISATION_ERROR_PX)]
    df = df[(df['yc_err'] > 0) & (df['yc_err'] < MAX_LOCALISATION_ERROR_PX)]
    df = df[(df['s_x'] > MIN_SIGMA_PX) & (df['s_x'] < MAX_SIGMA_PX)]
    df = df[(df['s_y'] > MIN_SIGMA_PX) & (df['s_y'] < MAX_SIGMA_PX)]
    df = df[df['s_x_err'] < MAX_SIGMA_ERROR_PX]
    df = df[df['s_y_err'] < MAX_SIGMA_ERROR_PX]
    df = df[df['photons'] > MIN_PHOTONS]
    return df.reset_index(drop=True)


example_fov = prime95b_files[0]
raw_df = IO.read_h5_database(str(example_fov))
print(f'[{example_fov.name}] {len(raw_df):,} raw localisations')

filt_df = filter_prime95b_locs(raw_df)
print(f'[{example_fov.name}] {len(filt_df):,} after filtering ({len(filt_df)/len(raw_df):.1%})')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].hist(raw_df['chi_sqr'], 100, alpha=0.5, label='raw')
axs[0].hist(filt_df['chi_sqr'], 100, alpha=0.5, label='filtered')
axs[0].set_xlabel('chi_sqr'); axs[0].legend()
axs[1].hist(raw_df['photons'], 100, range=(0, 20000), alpha=0.5, label='raw')
axs[1].hist(filt_df['photons'], 100, range=(0, 20000), alpha=0.5, label='filtered')
axs[1].set_xlabel('photons'); axs[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── AIM undrift preview ─────────────────────────────────────────────────────────
info = [{
    'Width': WIDTH, 'Height': HEIGHT,
    'Frames': int(filt_df['frame'].max()),
    'Pixelsize': PIXEL_SIZE_NM,
}]

drift_corrector = DCF.Drift_Correction_Functions()
corrected_locs, drift_result = drift_corrector.undrift(
    locs=filt_df.to_records(index=False),
    info=info,
    method='aim',
    segmentation=AIM_SEGMENTATION,
    intersect_d=AIM_INTERSECT_D,
    roi_r=AIM_ROI_R,
)
corrected_df = pd.DataFrame(corrected_locs)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(drift_result.drift_x * PIXEL_SIZE_NM, label='drift x')
ax.plot(drift_result.drift_y * PIXEL_SIZE_NM, label='drift y')
ax.set_xlabel('segment'); ax.set_ylabel('drift / nm'); ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Mono HDBSCAN linking helper ─────────────────────────────────────────────────
# Mirrors src/clustering/hdbscan_clusterer.py's HDBSCANMixin.extract_single_molecules_HDBSCAN
# exactly (same X construction, same cluster_selection_epsilon formula, same HDBSCAN
# call), but skips filter_quality_localisations (colour-hardcoded, see title cell)
# and reuses SM_E.average_parameters() directly for the per-cluster averaging step.

def extract_single_molecules_HDBSCAN_mono(df, min_cluster_size=10):
    if len(df) < min_cluster_size:
        return pd.DataFrame(), pd.DataFrame()

    X = np.vstack([df['xc'], df['yc']]).T
    loc_precision = 0.5 * (np.mean(df['xc_err']) + np.mean(df['yc_err']))

    hdb = HDBSCAN(min_cluster_size=min_cluster_size, cluster_selection_epsilon=loc_precision)
    hdb.fit(X)
    labels = hdb.labels_

    assigned_mask = labels >= 0
    df_assigned = df[assigned_mask].copy()
    labels_assigned = labels[assigned_mask]
    df_assigned['molecular_index'] = labels_assigned

    single_molecule_db = SM_E.average_parameters(df_assigned, labels_assigned)
    single_molecule_db['molecular_index'] = single_molecule_db.index
    return single_molecule_db, df_assigned


print('Mono HDBSCAN linking helper defined.')


In [ ]:
single_molecule_db, single_frame_db = extract_single_molecules_HDBSCAN_mono(
    corrected_df, min_cluster_size=MIN_CLUSTER_SIZE,
)
print(f'[{example_fov.name}] {len(single_molecule_db)} linked single molecules from '
      f'{len(single_frame_db)} assigned localisations (of {len(corrected_df)} undrifted)')

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(single_frame_db['xc'], single_frame_db['yc'], s=1, alpha=0.3,
           c=single_frame_db['molecular_index'], cmap='tab20')
ax.scatter(single_molecule_db['xc'], single_molecule_db['yc'], s=20, marker='+', color='black')
ax.set_aspect('equal')
ax.set_title(f'{example_fov.name}\n{len(single_molecule_db)} molecules')
plt.tight_layout()
plt.show()


### Full batch — run filter/undrift/link over every FOV

In [ ]:
# ── Full fit — every Prime95B FOV ───────────────────────────────────────────────
for i_fov, fov_path in enumerate(prime95b_files):
    sm_path     = fov_path.with_name(fov_path.name.replace('_nocolour_fit.h5', '_linked_sm.h5'))
    frames_path = fov_path.with_name(fov_path.name.replace('_nocolour_fit.h5', '_linked_frames.h5'))
    if sm_path.exists() and frames_path.exists():
        print(f'[{i_fov + 1:3d}/{len(prime95b_files)}] {fov_path.name}  (skipped, already linked)')
        continue

    raw_df  = IO.read_h5_database(str(fov_path))
    filt_df = filter_prime95b_locs(raw_df)
    if len(filt_df) < MIN_CLUSTER_SIZE:
        print(f'[{i_fov + 1:3d}/{len(prime95b_files)}] {fov_path.name}  '
              f'only {len(filt_df)} locs after filtering, skipping')
        continue

    info = [{
        'Width': WIDTH, 'Height': HEIGHT,
        'Frames': int(filt_df['frame'].max()),
        'Pixelsize': PIXEL_SIZE_NM,
    }]
    drift_corrector = DCF.Drift_Correction_Functions()
    corrected_locs, drift_result = drift_corrector.undrift(
        locs=filt_df.to_records(index=False), info=info, method='aim',
        segmentation=AIM_SEGMENTATION, intersect_d=AIM_INTERSECT_D, roi_r=AIM_ROI_R,
    )
    corrected_df = pd.DataFrame(corrected_locs)

    single_molecule_db, single_frame_db = extract_single_molecules_HDBSCAN_mono(
        corrected_df, min_cluster_size=MIN_CLUSTER_SIZE,
    )

    IO.write_h5_database(single_molecule_db, str(sm_path), normalise_photons=False)
    IO.write_h5_database(single_frame_db, str(frames_path), normalise_photons=False)

    print(f'[{i_fov + 1:3d}/{len(prime95b_files)}] {fov_path.name}  '
          f'{len(raw_df):,} -> {len(filt_df):,} filtered -> '
          f'{len(single_molecule_db)} linked molecules')

print('[Prime95B] Post-analysis done.')
